# Lesson 2: Vectors & Matrices

*ML Academy — companion notebook*

Lists and tables of numbers, the dot product, distances, and why predictions are matrix multiplications.

Run each cell in order with **Shift+Enter**. The explanations, maths and worked examples are in `lesson-02.html`; this notebook contains every code example from the lesson so you can run and modify it.

Requirements: `pip install numpy pandas scikit-learn matplotlib`

## 11. Doing it in Python with NumPy

In [1]:
import numpy as np

a = np.array([2, 3, 1])
b = np.array([4, -1, 5])
print(a + b)             # [6 2 6]      addition
print(3 * a)             # [6 9 3]      scalar multiplication
print(a @ b)             # 10           dot product
print(np.linalg.norm(np.array([3, 4])))   # 5.0   length

X = np.array([[50, 1, 2],
              [80, 2, 5],
              [120, 3, 12],
              [150, 4, 3]])
w = np.array([3, 20, -5])
print(X.shape)           # (4, 3)  4 examples, 3 features
print(X @ w)             # [160 255 360 515]   all predictions at once
print(X.T.shape)         # (3, 4)  transpose

[6 2 6]
[6 9 3]
10
5.0
(4, 3)
[160 255 360 515]
(3, 4)


## 14. Going deeper: the linear algebra behind ML algorithms

In [2]:
import numpy as np

v = np.array([3.0, -4.0, 1.0])
print("L1 norm (sum of sizes):    ", np.linalg.norm(v, 1))
print("L2 norm (straight line):   ", round(np.linalg.norm(v), 3))
print("L-infinity (biggest entry):", np.linalg.norm(v, np.inf))

M = np.array([[1.0, 2.0], [3.0, 4.0]])
print("Frobenius norm of M:       ", round(np.linalg.norm(M), 3))

L1 norm (sum of sizes):     8.0
L2 norm (straight line):    5.099
L-infinity (biggest entry): 4.0
Frobenius norm of M:        5.477


In [3]:
# Three features, but the third is the sum of the first two
X = np.array([[1, 2, 3],
              [2, 0, 2],
              [0, 1, 1],
              [3, 1, 4]], dtype=float)
print("columns:", X.shape[1], " rank:", np.linalg.matrix_rank(X))
print("det(XᵀX):", round(np.linalg.det(X.T @ X), 6))

A = np.array([[2.0, 1.0], [1.0, 3.0]])
print("det(A):", round(np.linalg.det(A), 3), "-> A scales areas by this factor")

columns: 3  rank: 2
det(XᵀX): 0.0
det(A): 5.0 -> A scales areas by this factor


In [4]:
vals, vecs = np.linalg.eigh(A)            # eigh: for symmetric matrices
print("eigenvalues:", vals.round(3))
print("eigenvectors (columns):")
print(vecs.round(3))
v1 = vecs[:, 1]
print("A @ v:     ", (A @ v1).round(3))
print("lambda * v:", (vals[1] * v1).round(3))
print("dot product of the two eigenvectors:", abs(round(vecs[:, 0] @ vecs[:, 1], 10)))

eigenvalues: [1.382 3.618]
eigenvectors (columns):
[[-0.851  0.526]
 [ 0.526  0.851]]
A @ v:      [1.902 3.078]
lambda * v: [1.902 3.078]
dot product of the two eigenvectors: 0.0


In [5]:
# A ratings table: 50 users × 40 films, built from 3 hidden "tastes" plus noise
rng = np.random.default_rng(0)
R = rng.normal(size=(50, 3)) @ rng.normal(size=(3, 40)) + rng.normal(0, 0.3, size=(50, 40))
U, s, Vt = np.linalg.svd(R, full_matrices=False)
print("largest singular values:", s[:6].round(1))
for k in [1, 2, 3, 5, 10]:
    approx = U[:, :k] @ np.diag(s[:k]) @ Vt[:k]          # keep only the k largest
    err = np.linalg.norm(R - approx) / np.linalg.norm(R)
    print(f"rank {k:2d}: stores {k * (50 + 40 + 1):4d} of 2000 numbers, relative error {err:.3f}")

largest singular values: [54.9 40.6 38.5  3.9  3.6  3.4]
rank  1: stores   91 of 2000 numbers, relative error 0.722
rank  2: stores  182 of 2000 numbers, relative error 0.510
rank  3: stores  273 of 2000 numbers, relative error 0.157
rank  5: stores  455 of 2000 numbers, relative error 0.142
rank 10: stores  910 of 2000 numbers, relative error 0.110


In [6]:
rng = np.random.default_rng(2)
n = 100
size = rng.uniform(50, 150, n)
size2 = size + rng.normal(0, 0.5, n)                 # a second, almost identical measurement
Xc = np.column_stack([size, size2]); Xc = Xc - Xc.mean(axis=0)
y = 0.6 * Xc[:, 0] + rng.normal(0, 5, n)
y_nudged = y + rng.normal(0, 1, n)                   # a small change in the targets

G = Xc.T @ Xc
print("condition number of XᵀX:", round(np.linalg.cond(G)))
print("weights:            ", np.linalg.solve(G, Xc.T @ y).round(2))
print("after a tiny change:", np.linalg.solve(G, Xc.T @ y_nudged).round(2))
Gr = G + 100 * np.eye(2)                             # ridge: add λI with λ = 100
print("condition number with ridge:", round(np.linalg.cond(Gr)))
print("ridge weights:      ", np.linalg.solve(Gr, Xc.T @ y).round(2))
print("after a tiny change:", np.linalg.solve(Gr, Xc.T @ y_nudged).round(2))

condition number of XᵀX: 14469
weights:             [ 0.96 -0.36]
after a tiny change: [ 1.2  -0.61]
condition number with ridge: 1462
ridge weights:       [0.37 0.23]
after a tiny change: [0.39 0.21]


In [7]:
# Check the matrix-calculus rule  ∇_w ||Xw − y||² = 2 Xᵀ(Xw − y)  with finite differences
rng = np.random.default_rng(3)
Xg = rng.normal(size=(5, 3)); yg = rng.normal(size=5); w = rng.normal(size=3)
f = lambda w: np.sum((Xg @ w - yg) ** 2)
formula = 2 * Xg.T @ (Xg @ w - yg)
h = 1e-6
numeric = np.array([(f(w + h * e) - f(w - h * e)) / (2 * h) for e in np.eye(3)])
print("formula:", formula.round(4))
print("numeric:", numeric.round(4))

formula: [-13.3106  19.8282   2.4238]
numeric: [-13.3106  19.8282   2.4238]


## Running example: Chennai Homes

The same 12-house dataset used in every lesson (`data/houses.csv` in the course folder).

In [8]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/houses.csv")
X = df[["size_m2", "bedrooms", "dist_km", "age_yrs"]].to_numpy()
print(X.shape)                                   # 12 houses x 4 features
a, b = X[0], X[1]                                # house 1 and house 2 as vectors
print(a, b, round(np.linalg.norm(a - b), 2))     # their Euclidean distance
w = np.array([0.6, 2.0, -1.2, -0.5])             # some made-up weights
print((X @ w)[:3].round(1))                      # predictions for the first 3 houses

(12, 4)
[65  1  2  5] [90  2  3 12] 26.0
[36.1 48.4 63.2]


## Coding challenges

Each cell below has **one missing line**, marked `# ← your code here` (or `# ← replace`). Write the line, run the cell, and the check at the bottom tells you whether you got it right. Stuck? A wrong answer prints a hint, and the solutions are at the very end of this notebook.

In [9]:
# Checker used by the challenges below — run this cell first.
import numpy as np
def check(got, want, hint='', atol=1e-6):
    try:
        ok = got is not None and np.shape(got) == np.shape(want) and np.allclose(np.asarray(got, dtype=float), np.asarray(want, dtype=float), rtol=1e-4, atol=atol)
    except Exception:
        ok = False
    print('✅ Correct!' if ok else '❌ Not yet. Hint: ' + hint)


### Challenge 1: Cosine similarity

Compute the cosine of the angle between a and b.

In [10]:
import numpy as np
a = np.array([1.0, 2.0, 2.0])
b = np.array([2.0, 0.0, 1.0])
cos_sim = None  # ← your code here
check(cos_sim, 0.5962847939999439, hint='Dot product divided by the product of the two lengths (np.linalg.norm).')

❌ Not yet. Hint: Dot product divided by the product of the two lengths (np.linalg.norm).


### Challenge 2: Predictions for every row

Compute the prediction X·w for all three examples in one line.

In [11]:
import numpy as np
X = np.array([[1, 2], [3, 4], [5, 6]])
w = np.array([0.5, -1.0])
preds = None  # ← your code here
check(preds, [-1.5, -2.5, -3.5], hint='Matrix-vector product: X @ w.')

❌ Not yet. Hint: Matrix-vector product: X @ w.


## Your turn

Try the **practice exercises** at the end of the lesson page. Many of them ask you to change one of the cells above and re-run it.

## Challenge solutions

*Spoilers — try the challenges first!*

**1. Cosine similarity** — the missing line:

```python
cos_sim = a @ b / (np.linalg.norm(a) * np.linalg.norm(b))
```

**2. Predictions for every row** — the missing line:

```python
preds = X @ w
```